In [3]:
!nvidia-smi -L

GPU 0: NVIDIA H200 (UUID: GPU-d828dffb-766b-5b29-d4ef-967798fac01a)
GPU 1: NVIDIA H200 (UUID: GPU-0a87e2b3-152c-ea81-98ca-5cc011464a8a)
GPU 2: NVIDIA H200 (UUID: GPU-4511012b-7300-a0cb-1cff-7d7a14de1357)
GPU 3: NVIDIA H200 (UUID: GPU-0138eb1f-2bc3-ee30-9b9e-b2f2e7f74692)
GPU 4: NVIDIA H200 (UUID: GPU-cf25233a-6aac-3e6a-3cce-1b9100b54dc0)
GPU 5: NVIDIA H200 (UUID: GPU-d901ddb1-cfdd-16e9-2a5f-f1eaff8aa73f)
  MIG 3g.71gb     Device  0: (UUID: MIG-44e4992a-c43a-5c9c-a696-ba447b29b04a)
  MIG 2g.35gb     Device  1: (UUID: MIG-f87cc43f-5fcc-565f-9356-6193344d22fc)
  MIG 1g.18gb     Device  2: (UUID: MIG-487d15d4-05ae-5916-9c83-a8d0c313efc4)
  MIG 1g.18gb     Device  3: (UUID: MIG-87595212-e000-5cf4-ad2c-2a60e6e789c1)
GPU 6: NVIDIA H200 (UUID: GPU-b6d7aef0-fec8-6e2d-1863-e6fdbed075b4)
GPU 7: NVIDIA H200 (UUID: GPU-189bb538-15a4-4eac-8739-33807df06028)
  MIG 3g.71gb     Device  0: (UUID: MIG-240c1e89-b797-5e56-9ad8-69c76419e0a8)
  MIG 2g.35gb     Device  1: (UUID: MIG-684087f3-5bfe-5277-9268-9f

In [8]:
import sys
import pandas as pd
from utilites import timer_utility
import numpy as np
import os

In [9]:
# !{sys.executable} -m pip install cudf-cu12 cuml-cu12 cupy-cuda12x
# !{sys.executable} -m pip install pandas
# !{sys.executable} -m pip install cudf-cu12 --no-user
# !pip install cudf-cu12 

In [10]:
def expand_csv(input_file, target_rows, output_file, seed=42):
    np.random.seed(seed)
    df = pd.read_csv(input_file)
    n_rep  = (target_rows // len(df)) + 1
    df_out = pd.concat([df] * n_rep, ignore_index=True).head(target_rows).copy()
    n = len(df_out)
    df_out['transaction_id'] = np.arange(1, n + 1)
    df_out['revenue']   = (df_out['revenue'] * np.random.uniform(0.95, 1.05, n)).round(2)
    df_out['user_id']   = np.random.randint(1, max(n // 10, 2), n)
    df_out['timestamp'] = pd.date_range('2024-01-01', periods=n, freq='30s').strftime('%Y-%m-%d %H:%M:%S')
    df_out.to_csv(output_file, index=False)
    mb = os.path.getsize(output_file) / 1024**2
    print(f"✅ {output_file:40s} → {n:>10,} baris | {mb:.1f} MB")


INPUT = 'benchmark_transactions_1000.csv'
expand_csv(INPUT,    10_000, 'data_10k.csv')
expand_csv(INPUT,   100_000, 'data_100k.csv')
expand_csv(INPUT,   500_000, 'data_500k.csv')
expand_csv(INPUT, 1_000_000, 'data_1M.csv')

✅ data_10k.csv                             →     10,000 baris | 1.2 MB
✅ data_100k.csv                            →    100,000 baris | 11.8 MB
✅ data_500k.csv                            →    500,000 baris | 59.8 MB
✅ data_1M.csv                              →  1,000,000 baris | 119.9 MB


### read csv with pandas

In [34]:
@timer_utility.timer
def read_csv_cpu(path:str) -> pd:
    try:
        return pd.read_csv(path)
    except:
        raise Exception("Gagal membaca file csv")

#### load using CPU

In [35]:
read_csv_cpu("data_1M.csv").head()

[timer] 'read_csv_cpu' → 1.5830s


,transaction_id,user_id,product_id,category,region,payment_type,status,brand,revenue,quantity,discount_pct,shipping_cost,cost_price,rating,timestamp,promo_code
0,1,18830,29329,Food,Bali,Bank Transfer,completed,Brand_D,633487.13,8,0.2976,40886.98,177372.21,2.0,2024-01-01 00:00:00,NaN
1,2,32325,46480,Travel,Palembang,Bank Transfer,completed,Brand_D,60478.86,16,0.1824,16761.60,183681.69,3.0,2024-01-01 00:00:30,NaN
2,3,9053,42892,Electronics,Medan,Credit Card,pending,Brand_B,530104.20,13,0.0027,33443.55,415376.04,4.0,2024-01-01 00:01:00,NaN
3,4,87877,20104,Home,Surabaya,Paylater,completed,Brand_E,7809.11,18,0.2805,25189.18,71708.12,4.0,2024-01-01 00:01:30,NaN
4,5,54179,39892,Electronics,Medan,Bank Transfer,completed,Brand_D,22999.27,10,0.4483,32319.97,468890.50,3.0,2024-01-01 00:02:00,SALE30


#### load using GPU

In [36]:
import cudf
import cupy as cp
print(f"cuDF version: {cudf.__version__}")
print(f"GPU: {cp.cuda.Device(0).attributes}")

cuDF version: 26.06.00
GPU: {'AsyncEngineCount': 3, 'CanFlushRemoteWrites': 0, 'CanMapHostMemory': 1, 'CanUseHostPointerForRegisteredMem': 1, 'ClockRate': 1980000, 'ComputeMode': 0, 'ComputePreemptionSupported': 1, 'ConcurrentKernels': 1, 'ConcurrentManagedAccess': 1, 'CooperativeLaunch': 1, 'CooperativeMultiDeviceLaunch': 1, 'DirectManagedMemAccessFromHost': 0, 'EccEnabled': 1, 'GPUDirectRDMAFlushWritesOptions': 1, 'GPUDirectRDMASupported': 1, 'GPUDirectRDMAWritesOrdering': 100, 'GlobalL1CacheSupported': 1, 'GlobalMemoryBusWidth': 3072, 'GpuOverlap': 1, 'HostNativeAtomicSupported': 0, 'HostRegisterReadOnlySupported': 1, 'HostRegisterSupported': 1, 'Integrated': 0, 'IsMultiGpuBoard': 0, 'KernelExecTimeout': 0, 'L2CacheSize': 30801920, 'LocalL1CacheSupported': 1, 'ManagedMemory': 1, 'MaxBlockDimX': 1024, 'MaxBlockDimY': 1024, 'MaxBlockDimZ': 64, 'MaxBlocksPerMultiprocessor': 32, 'MaxGridDimX': 2147483647, 'MaxGridDimY': 65535, 'MaxGridDimZ': 65535, 'MaxPitch': 2147483647, 'MaxRegistersP

In [41]:
@timer_utility.timer
def read_csv_gpu(path:str) -> cudf.DataFrame:
    try:
        return cudf.read_csv(path)
    except:
        raise Exception("Gagal membaca file csv")

In [42]:
df_cu = read_csv_gpu("data_1M.csv").head()

[timer] 'read_csv_gpu' → 0.0804s


In [43]:
df_cu

,transaction_id,user_id,product_id,category,region,payment_type,status,brand,revenue,quantity,discount_pct,shipping_cost,cost_price,rating,timestamp,promo_code
0,1,18830,29329,Food,Bali,Bank Transfer,completed,Brand_D,633487.13,8,0.2976,40886.98,177372.21,2.0,2024-01-01 00:00:00,None
1,2,32325,46480,Travel,Palembang,Bank Transfer,completed,Brand_D,60478.86,16,0.1824,16761.60,183681.69,3.0,2024-01-01 00:00:30,None
2,3,9053,42892,Electronics,Medan,Credit Card,pending,Brand_B,530104.20,13,0.0027,33443.55,415376.04,4.0,2024-01-01 00:01:00,None
3,4,87877,20104,Home,Surabaya,Paylater,completed,Brand_E,7809.11,18,0.2805,25189.18,71708.12,4.0,2024-01-01 00:01:30,None
4,5,54179,39892,Electronics,Medan,Bank Transfer,completed,Brand_D,22999.27,10,0.4483,32319.97,468890.50,3.0,2024-01-01 00:02:00,SALE30
